# Training-free GMM sampling: TSI vs DSI vs CVSI

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from dem.models.components.clipper import Clipper
from dem.models.components.cvsi import get_a2_b2
from dem.models.components.sdes import ReverseSDE
from dem.models.components.sde_integration import integrate_sde
from dem.models.components.optimal_transport import wasserstein
from vp_utils import build_schedule
from gmm_toy import GMMEnergy                       # normalized GMM (GMMNorm-style whitening)
from estimators import score_from_samples

# float64 for clean unit tests; the generation section can be switched to float32 on a GPU node.
torch.set_default_dtype(torch.float64)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------------ flags
SCHEDULE = "VP"     # "VE" (a=1) or "VP" (SNR/TV, a(t)->0). Estimator code is schedule-general.
TOY      = False    # True -> 2-D / few modes (CPU laptop); False -> 50-D / 30 modes (GPU node).
SMOKE    = False    # True -> tiny B/K/steps for a fast code smoke-test.

assert SCHEDULE in ("VE", "VP")
# SMOKE controls batch/step sizes; TOY controls the problem dimension.
CFG = dict(
    smoke=dict(B=64,   NT=8,  K_EXACT=1024, N_GEN=64,   T_STEPS=40, K_GEN=32, N_SEEDS=3),
    full =dict(B=1000, NT=12, K_EXACT=200,  N_GEN=1000, T_STEPS=64, K_GEN=4,  N_SEEDS=2),
)["smoke" if SMOKE else "full"]

torch.manual_seed(0)
print("torch", torch.__version__, "| device", device,
      "| SCHEDULE", SCHEDULE, "| TOY", TOY, "| SMOKE", SMOKE)
print("config:", CFG)

## 1. Instantiate the GMM and the noise schedule

`make_gmm` mirrors `GMMNorm.__init__`; `GMMEnergy(normalize=True)` applies the same scalar
whitening. `build_schedule` returns the schedule, its reverse-SDE time domain and the prior
std. The printed per-dim std should read ~1.

In [ ]:
def make_gmm(D, M, seed=0, mean_scale=10.0, df_scale=2.0, device="cpu",
             isotropic=False):
    """Normalized GMM built with the GMMNorm recipe, whitened to per-dim var ~ 1.

    isotropic: False | True (whiten the mixture so total cov = I -- what VP needs) |
    "components" (spherical components only). See gmm_toy.GMMEnergy.
    """
    g = torch.Generator(device="cpu").manual_seed(seed)
    weights = torch.rand(M, generator=g)
    means = torch.randn(M, D, generator=g) * (mean_scale * D ** 0.5)
    df = max(D, int(D * df_scale))
    covs = torch.stack([(lambda z: z.T @ z)(torch.randn(df, D, generator=g)) for _ in range(M)])
    # GMMEnergy(normalize=True) applies the same scalar whitening as GMMNorm.
    return GMMEnergy(means.to(device), covs.to(device), weights.to(device),
                     normalize=True, isotropic=isotropic)

D, M = (2, 6) if TOY else (50, 30)
# ISOTROPIC: False (default) | True (whiten the mixture to total cov = I) |
# "components" (spherical Wishart components only; mixture stays anisotropic because
# the per-dim spread is dominated by the mean scatter).
# VP's constant TV preserves a^2 Var(x0)+b^2 per *direction*, which a single scalar
# only achieves on isotropic data -- set True to make that assumption exact.
ISOTROPIC = False
gmm = make_gmm(D, M, seed=0, device=device, isotropic=ISOTROPIC)

# VP ignores sigma_min/sigma_max (SNR/TV is fixed). Data is already unit-variance
# from the whitening above, so scale=1.0.
# GEN_START_T: reverse-SDE start (None = schedule default; VE 1.0, VP t_max=0.99).
GEN_START_T = None

spec = build_schedule(SCHEDULE, sigma_min=0.001, sigma_max=50.0, scale=1.0,
                      gen_start=GEN_START_T)
noise = spec.noise
print(spec.summary())

# t-grid over the schedule's active range (high-t end = spec.gen_start: VE 1.0, VP t_max).
t_lo = 0.05 if SCHEDULE == "VE" else noise.t_min
t_hi = spec.gen_start
# tgrid must live on `device`: t becomes the a(t) scaling applied to the TSI/DSI scores.
tgrid = torch.linspace(float(t_lo), float(t_hi), CFG["NT"], device=device)

def ab_of(t):
    """(a^2, b^2) scalars at time t on `device`."""
    a2, b2 = get_a2_b2(torch.as_tensor(float(t), device=device).reshape(1), noise)
    return a2.reshape(()), b2.reshape(())

# TSI-only norm-clip on the energy-gradient branch, as in idem_score_cvsi_fn.
clipper = Clipper(True, False, max_score_norm=100.0, min_log_reward=None)

_smp = gmm.sample(4000)
print(f"D={D}  M={M}  prior_std={spec.prior_std:.3f}  gen: t {spec.gen_start:.4g} -> {spec.gen_end:.4g}")
print(f"data per-dim std ~ {_smp.std(0).mean().item():.2f} (normalized ~1)   "
      f"||mu_i - mu_j|| range: {torch.pdist(gmm.means).min().item():.2f}"
      f"..{torch.pdist(gmm.means).max().item():.2f}")
print(f"mode occupancy of a fresh sample: "
      f"{torch.bincount(gmm.nearest_mode(_smp), minlength=M).float().div(len(_smp)).cpu().numpy().round(2)}")

from style import PAPER_COLORS as EST_COLOR
ESTIMATORS = ["tsi", "dsi", "cvsi"]


## A. Estimator accuracy with exact posterior samples

With exact draws the only error is Monte-Carlo variance, so all three should be small with
CVSI at or below min(TSI, DSI) at every `t`. Under VP the `1/a(t)` factor inflates TSI at
high `t`; under VE (`a=1`) all three stay close.

In [ ]:
K_EXACT = CFG["K_EXACT"]
B = CFG["B"]
torch.manual_seed(1)
x0_data = gmm.sample(B)                                      # anchor points
rel_err = {e: [] for e in ESTIMATORS}
for t in tgrid:
    a2, b2 = ab_of(t)
    tt = t.repeat(B)
    xt = a2.sqrt() * x0_data + b2.sqrt() * torch.randn_like(x0_data)
    s_star = gmm.marginal_score(xt, a2, b2)
    den = s_star.pow(2).sum(-1).sqrt().clamp_min(1e-9)
    x0e = gmm.sample_posterior_exact(xt, a2, b2, K_EXACT)
    w = torch.full((K_EXACT, B, 1), 1.0 / K_EXACT, device=device)
    h_t = torch.full((K_EXACT, B, 1), float(b2), device=device)
    for e in ESTIMATORS:
        s = score_from_samples(tt, xt, x0e, w, h_t, gmm, noise, estimator=e)
        rel_err[e].append(((s - s_star).pow(2).sum(-1).sqrt() / den).mean().item())

plt.figure(figsize=(6.5, 4))
for e in ESTIMATORS:
    plt.plot(tgrid.cpu(), rel_err[e], "o-", color=EST_COLOR[e], label=e.upper())
plt.yscale("log"); plt.xlabel("t"); plt.ylabel(r"rel. error $\|\hat s - s^\star\|/\|s^\star\|$")
plt.title(f"A. Estimators vs analytic score, EXACT posterior ({SCHEDULE}, d={D}, K={K_EXACT})")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
for e in ESTIMATORS:
    print(f"{e.upper():4s} rel-err: t={float(tgrid[0]):.3f} -> {rel_err[e][0]:.2e}   "
          f"t={float(tgrid[-1]):.3f} -> {rel_err[e][-1]:.2e}")
print("\nPASS if all three are small and CVSI <= min(TSI, DSI) at every t.")

## B. End-to-end reverse SDE, seed-averaged

Generation driven by each estimator on exact posterior draws, through `ReverseSDE` integrated
from `spec.gen_start` to `spec.gen_end`. Metrics: W2 to true samples, mean/cov error, mode
coverage. Under VP constant TV=1 keeps the marginal std ~1 at every `t`, so sample std is
uninformative there: read W2 and mode coverage.

In [ ]:
N_GEN, T_STEPS, K_GEN, N_SEEDS = CFG["N_GEN"], CFG["T_STEPS"], CFG["K_GEN"], CFG["N_SEEDS"]

# analytic mixture moments (law of total covariance) for the mean/cov error
mu_mix = (gmm.w[:, None] * gmm.means).sum(0)
diff_mix = gmm.means - mu_mix
Sigma_mix = (gmm.w[:, None, None] *
             (gmm.covs + torch.einsum('md,me->mde', diff_mix, diff_mix))).sum(0)
ref = gmm.sample(2000)
prior_std, gen_start, gen_end = spec.prior_std, spec.gen_start, spec.gen_end


def make_score_fn(est):
    # closed-form posterior draws
    def exact_score_fn(t, x):
        a2, b2 = ab_of(t.reshape(-1)[0])
        x0e = gmm.sample_posterior_exact(x, a2, b2, K_GEN)
        w = torch.full((K_GEN, x.shape[0], 1), 1.0 / K_GEN, device=device)
        h_t = torch.full((K_GEN, x.shape[0], 1), float(b2), device=device)
        return score_from_samples(t, x, x0e, w, h_t, gmm, noise,
                                  estimator=est, clipper_tsi=clipper)
    return exact_score_fn


def generate(est, seed):
    torch.manual_seed(seed)
    sde = ReverseSDE(make_score_fn(est), noise)
    x_init = torch.randn(N_GEN, D, device=device) * prior_std
    traj = integrate_sde(sde, x_init, num_integration_steps=T_STEPS, energy_function=gmm,
                         diffusion_scale=1.0, no_grad=True,
                         start_time=gen_start, end_time=gen_end)
    return traj[-1].detach()

rows = []
gen_cache = {}
for e in ESTIMATORS:
    me, ce, w2, nm = [], [], [], []
    for s in range(N_SEEDS):
        g = generate(e, seed=s)
        ok = torch.isfinite(g).all(-1)
        g = g[ok] if ok.any() else g
        if s == 0:
            gen_cache[e] = g
        me.append((g.mean(0) - mu_mix).norm().item())
        ce.append((torch.cov(g.T) - Sigma_mix).norm().item())
        w2.append(wasserstein(g, ref[:len(g)], power=2))
        nm.append(len(torch.unique(gmm.nearest_mode(g))))
    agg = lambda v: (float(np.mean(v)), float(np.std(v)))
    rows.append((e.upper(), agg(me), agg(ce), agg(w2), agg(nm)))

print(f"{'ident':6s}{'mean_err':>15s}{'cov_err':>15s}{'W2':>15s}{'modes/'+str(M):>12s}")
print(f"(mean +- std over {N_SEEDS} seeds; exact posterior; schedule={SCHEDULE})")
for e, me, ce, w2, nm in rows:
    print(f"{e:6s}"
          f"{me[0]:10.3f}+-{me[1]:4.2f}"
          f"{ce[0]:10.3f}+-{ce[1]:4.2f}"
          f"{w2[0]:10.3f}+-{w2[1]:4.2f}"
          f"{nm[0]:8.1f}+-{nm[1]:3.1f}")
print(f"\ntrue mixture mean norm={mu_mix.norm().item():.3f}  ({M} modes; 'modes' near {M} = full coverage)")
print("Read identity differences as real only if they exceed ~1 std.")

### B. PCA projection of true vs generated samples

In [ ]:
# PCA basis from the true samples (2-D is projected onto itself)
ref_c = ref - ref.mean(0)
U, S, Vh = torch.linalg.svd(ref_c, full_matrices=False)
P = Vh[:2].T                                                # (D,2)
def proj(x): return ((x - ref.mean(0)) @ P).cpu()
means_p = proj(gmm.means)

fig, ax = plt.subplots(1, len(ESTIMATORS), figsize=(5.0 * len(ESTIMATORS), 4.6), sharex=True, sharey=True)
rp = proj(ref[:1500])
for a, e in zip(ax, ESTIMATORS):
    a.scatter(rp[:, 0], rp[:, 1], s=6, alpha=0.15, c="k", label="true")
    g = proj(gen_cache[e])
    a.scatter(g[:, 0], g[:, 1], s=10, alpha=0.5, c=EST_COLOR[e], label=e.upper())
    a.scatter(means_p[:, 0], means_p[:, 1], marker="*", s=220, c="red", edgecolor="k",
              label="mode means", zorder=5)
    a.set_title(e.upper()); a.set_xlabel("PC1"); a.legend(fontsize=8)
ax[0].set_ylabel("PC2")
plt.suptitle(f"PCA of true GMM ({SCHEDULE}, d={D}) vs generated samples", y=1.02)
plt.tight_layout(); plt.show()
